In [ ]:
!pip install kaggle==1.5.16

In [ ]:
! chmod 600 .kaggle/kaggle.json

In [ ]:
! kaggle competitions download Walmart-Recruiting-Store-Sales-Forecasting

In [ ]:
! unzip Walmart-Recruiting-Store-Sales-Forecasting.zip

In [ ]:
! unzip features.csv.zip
! unzip test.csv.zip
! unzip train.csv.zip

# TFT — Walmart Store Sales Forecasting

In [ ]:
!pip install "numpy<2" "torchvision==0.17.0" "torch==2.2.0" "neuralforecast==1.7.4" optuna mlflow dagshub wandb -q --force-reinstall

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os
import wandb
import mlflow
import mlflow.pyfunc

from neuralforecast import NeuralForecast
from neuralforecast.models import TFT
from neuralforecast.auto import AutoTFT
from neuralforecast.losses.pytorch import MAE
from ray import tune

WANDB_ENTITY  = 'ikvas22-free-university-of-tbilisi'
WANDB_PROJECT = 'Walmart Weekly Sales Prediction'
WANDB_GROUP   = 'TFT'

MLFLOW_EXPERIMENT = 'TFT_Training'
MLFLOW_MODEL_NAME = 'tft_walmart_best'

TRAIN_PATH    = 'train.csv'
FEATURES_PATH = 'features.csv'
STORES_PATH   = 'stores.csv'

H          = 4
INPUT_SIZE = 52
N_TRIALS   = 5
VAL_START  = '2012-04-01'

wandb.login()

print('Setup OK')

## 1. Pre-processing

In [ ]:
run = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'preprocessing',
    name     = 'TFT_Preprocessing_MoreTrials',
)

train_raw    = pd.read_csv(TRAIN_PATH,    parse_dates=['Date'])
features_raw = pd.read_csv(FEATURES_PATH, parse_dates=['Date'])
stores_raw   = pd.read_csv(STORES_PATH)

df = (
    train_raw
    .merge(features_raw, on=['Store', 'Date', 'IsHoliday'], how='left')
    .merge(stores_raw,   on=['Store'],                       how='left')
)

wandb.log({
    'raw_rows' : df.shape[0],
    'raw_cols' : df.shape[1],
    'stores'   : df['Store'].nunique(),
    'depts'    : df['Dept'].nunique(),
    'date_min' : str(df['Date'].min().date()),
    'date_max' : str(df['Date'].max().date()),
})

null_pct = df.isnull().mean().mul(100).round(2)
null_df  = null_pct[null_pct > 0].reset_index()
null_df.columns = ['column', 'null_pct']
wandb.log({'null_percentages': wandb.Table(dataframe=null_df)})
print('Nulls (%):')
print(null_df.to_string(index=False))

# ── Fill nulls in covariate columns ───────────────────────────
# TFT uses covariates so we need to handle nulls in feature columns.
# MarkDown columns have ~65% nulls — fill with 0 (no markdown active)
markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']
df[markdown_cols] = df[markdown_cols].fillna(0)
# CPI and Unemployment: fill with store-level forward fill then global median
df[['CPI', 'Unemployment']] = (
    df.groupby('Store')[['CPI', 'Unemployment']]
    .transform(lambda x: x.ffill().bfill())
)
df[['CPI', 'Unemployment']] = df[['CPI', 'Unemployment']].fillna(df[['CPI', 'Unemployment']].median())

wandb.log({
    'nulls_after_fill': int(df.isnull().sum().sum()),
})

# ── TFT-specific: add calendar covariates ─────────────────────
# TFT can consume known future covariates — calendar features are
# known in advance so they are the cleanest input for TFT.
df['week']        = df['Date'].dt.isocalendar().week.astype(int)
df['month']       = df['Date'].dt.month
df['is_holiday']  = df['IsHoliday'].astype(int)

# ── Format for neuralforecast ──────────────────────────────────
df['unique_id'] = df['Store'].astype(str) + '_' + df['Dept'].astype(str)

# Covariates TFT will use:
# - hist_exog: observed in the past only (markdowns, temperature, fuel)
# - futr_exog: known for the future too (calendar features, IsHoliday)
# - stat_exog: static per series (store type, store size)
# Replace these lines:
HIST_EXOG = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5',
             'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']
FUTR_EXOG = ['week', 'month', 'is_holiday']
STAT_EXOG = []   # dropped — static features require separate static_df in nf 1.7.4

# And in keep_cols, remove Type/Size:
keep_cols = ['unique_id', 'Date', 'Weekly_Sales'] + HIST_EXOG + FUTR_EXOG
df_nf = (
    df[keep_cols]
    .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    .sort_values(['unique_id', 'ds'])
    .reset_index(drop=True)
)

wandb.log({
    'hist_exog_cols' : HIST_EXOG,
    'futr_exog_cols' : FUTR_EXOG,
})

min_len    = INPUT_SIZE + H
series_len = df_nf.groupby('unique_id')['ds'].count()
valid_ids  = series_len[series_len >= min_len].index
dropped    = series_len[series_len < min_len].index.tolist()
df_nf      = df_nf[df_nf['unique_id'].isin(valid_ids)].reset_index(drop=True)

wandb.log({
    'total_series'   : len(series_len),
    'valid_series'   : len(valid_ids),
    'dropped_series' : len(dropped),
    'min_series_len' : int(series_len[valid_ids].min()),
    'max_series_len' : int(series_len[valid_ids].max()),
})

print(f'\nValid series : {len(valid_ids)}')
print(f'Dropped      : {len(dropped)} (shorter than {min_len} weeks)')

df_train = df_nf[df_nf['ds'] <  VAL_START].copy()
df_val   = df_nf[df_nf['ds'] >= VAL_START].copy()

wandb.log({
    'train_rows'     : len(df_train),
    'val_rows'       : len(df_val),
    'train_date_min' : str(df_train['ds'].min().date()),
    'train_date_max' : str(df_train['ds'].max().date()),
    'val_date_min'   : str(df_val['ds'].min().date()),
    'val_date_max'   : str(df_val['ds'].max().date()),
    'val_start'      : VAL_START,
    'horizon_weeks'  : H,
    'lookback_weeks' : INPUT_SIZE,
})

print(f'Train : {df_train["ds"].min().date()} → {df_train["ds"].max().date()}  ({len(df_train):,} rows)')
print(f'Val   : {df_val["ds"].min().date()} → {df_val["ds"].max().date()}  ({len(df_val):,} rows)')

wandb.finish()

## 2. Training

In [ ]:
def wmae(y_true: np.ndarray, y_pred: np.ndarray, is_holiday: np.ndarray) -> float:
    weights = np.where(is_holiday, 5.0, 1.0)
    return float(np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights))


def evaluate(nf: NeuralForecast, pred_col: str) -> tuple:
    preds   = nf.predict(df=df_val)
    eval_df = df_val.merge(
        preds.rename(columns={pred_col: 'y_pred'}),
        on=['unique_id', 'ds'], how='inner'
    )
    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['is_holiday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

df_full = pd.concat([df_train, df_val]).sort_values(['unique_id', 'ds']).reset_index(drop=True)

N_WINDOWS = 7

def evaluate_cv(nf: NeuralForecast, model_col: str) -> tuple:
    cv_df = nf.cross_validation(
        df        = df_full,      # full dataframe including exog columns
        n_windows = N_WINDOWS,
        step_size = H,
    )
    cv_df = cv_df.reset_index() if 'unique_id' not in cv_df.columns else cv_df

    eval_df = cv_df.merge(
        df_full[['unique_id', 'ds', 'is_holiday']],
        on=['unique_id', 'ds'], how='left'
    ).rename(columns={model_col: 'y_pred'})

    score_wmae = wmae(eval_df['y'].values, eval_df['y_pred'].values, eval_df['is_holiday'].values)
    score_mae  = float(np.mean(np.abs(eval_df['y'].values - eval_df['y_pred'].values)))
    return score_wmae, score_mae, eval_df

In [ ]:
run_baseline = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'TFT_Baseline_MoreTrials',
    config   = {
        'input_size'    : INPUT_SIZE,
        'h'             : H,
        'hidden_size'   : 64,
        'n_head'        : 4,
        'dropout'       : 0.1,
        'max_steps'     : 500,
        'learning_rate' : 1e-3,
        'batch_size'    : 32,
        'loss'          : 'MAE',
        'hist_exog'     : HIST_EXOG,
        'futr_exog'     : FUTR_EXOG,
        'eval'          : f'rolling_cv_{N_WINDOWS}_windows',
    }
)

baseline_model = TFT(
    h                     = H,
    input_size            = INPUT_SIZE,
    hidden_size           = 64,
    n_head                = 4,
    dropout               = 0.1,
    hist_exog_list        = HIST_EXOG,
    futr_exog_list        = FUTR_EXOG,
    loss                  = MAE(),
    max_steps             = 500,
    learning_rate         = 1e-3,
    batch_size            = 32,
    val_check_steps       = 50,
    start_padding_enabled = True,
    logger                = True,
)

nf_baseline = NeuralForecast(models=[baseline_model], freq='W-FRI')
baseline_wmae, baseline_mae, eval_baseline = evaluate_cv(nf_baseline, 'TFT')

wandb.log({
    'val_wmae' : baseline_wmae,
    'val_mae'  : baseline_mae,
})

print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Baseline MAE  : {baseline_mae:,.2f}')

wandb.finish()

In [ ]:
N_TRIALS = 30   # TFT is slow — keep trials modest

run_search = wandb.init(
    entity   = WANDB_ENTITY,
    project  = WANDB_PROJECT,
    group    = WANDB_GROUP,
    job_type = 'training',
    name     = 'TFT_HPSearch_MoreTrials',
    config   = {
        'n_trials'    : N_TRIALS,
        'h'           : H,
        'search_type' : 'random',
        'search_space': {
            'input_size'    : [26, 52, 104],
            'hidden_size'   : [32, 64, 128],
            'n_head'        : [2, 4],
            'learning_rate' : [1e-4, 5e-4, 1e-3],
            'batch_size'    : [32, 64, 128],
        }
    }
)

from itertools import product
import random
random.seed(42)

search_space = {
    'input_size'    : [26, 52, 104],
    'hidden_size'   : [32, 64, 128],
    'n_head'        : [2, 4],
    'learning_rate' : [1e-4, 5e-4, 1e-3],
    'batch_size'    : [32, 64, 128],
}

all_combos = list(product(*search_space.values()))
sampled    = random.sample(all_combos, min(N_TRIALS, len(all_combos)))
keys       = list(search_space.keys())

# Include the baseline config as trial 0 — guarantees search >= baseline
baseline_combo = (INPUT_SIZE, 64, 4, 1e-3, 32)
if baseline_combo not in sampled:
    sampled = [baseline_combo] + sampled[:-1]

best_wmae     = float('inf')
best_mae      = None
best_params   = None
nf_auto       = None
trial_results = []

for i, combo in enumerate(sampled):
    params = dict(zip(keys, combo))
    print(f'\n── Trial {i+1}/{len(sampled)}: {params}')

    model = TFT(
        h                     = H,
        input_size            = params['input_size'],
        hidden_size           = params['hidden_size'],
        n_head                = params['n_head'],
        dropout               = 0.1,
        hist_exog_list        = HIST_EXOG,
        futr_exog_list        = FUTR_EXOG,
        loss                  = MAE(),
        max_steps             = 1000,
        learning_rate         = params['learning_rate'],
        batch_size            = params['batch_size'],
        val_check_steps       = 50,
        start_padding_enabled = True,
    )

    nf = NeuralForecast(models=[model], freq='W-FRI')
    trial_wmae, trial_mae, _ = evaluate_cv(nf, 'TFT')

    print(f'   WMAE: {trial_wmae:,.2f}   MAE: {trial_mae:,.2f}')
    trial_results.append({
        'trial'         : i + 1,
        'input_size'    : params['input_size'],
        'hidden_size'   : params['hidden_size'],
        'n_head'        : params['n_head'],
        'learning_rate' : params['learning_rate'],
        'batch_size'    : params['batch_size'],
        'wmae'          : trial_wmae,
        'mae'           : trial_mae,
    })

    if trial_wmae < best_wmae:
        best_wmae   = trial_wmae
        best_mae    = trial_mae
        best_params = params
        nf_auto     = nf

trials_df = pd.DataFrame(trial_results)
_, _, eval_best = evaluate_cv(nf_auto, 'TFT')

wandb.log({
    'best_val_wmae'    : best_wmae,
    'best_val_mae'     : best_mae,
    'baseline_wmae'    : baseline_wmae,
    'wmae_improvement' : baseline_wmae - best_wmae,
    'best_params'      : str(best_params),
    'trials_completed' : len(sampled),
    'all_trials'       : wandb.Table(dataframe=trials_df),
})

print(f'\n════════════════════════════════════')
print(f'Best WMAE     : {best_wmae:,.2f}')
print(f'Baseline WMAE : {baseline_wmae:,.2f}')
print(f'Improvement   : {baseline_wmae - best_wmae:,.2f}')
print(f'Best params   : {best_params}')

# Prediction plots
sample_ids = eval_best['unique_id'].unique()[:6]
fig, axes  = plt.subplots(3, 2, figsize=(14, 10))
axes       = axes.flatten()

for ax, uid in zip(axes, sample_ids):
    history = df_train[df_train['unique_id'] == uid].tail(52)
    actual  = eval_best[eval_best['unique_id'] == uid]
    ax.plot(history['ds'], history['y'],     label='History', color='steelblue')
    ax.plot(actual['ds'],  actual['y'],      label='Actual',  color='black')
    ax.plot(actual['ds'],  actual['y_pred'], label='TFT',     color='tomato', linestyle='--')
    hol = actual[actual['is_holiday'] == 1]
    ax.scatter(hol['ds'], hol['y'], color='gold', zorder=5, s=40, label='Holiday')
    ax.set_title(f'Store-Dept {uid}', fontsize=10)
    ax.legend(fontsize=7)
    ax.tick_params(axis='x', rotation=30)

plt.suptitle('TFT Best Model — Validation Predictions', fontsize=13)
plt.tight_layout()
wandb.log({'prediction_plots': wandb.Image(fig)})
plt.show()

# Per-series WMAE
per_series = (
    eval_best
    .groupby('unique_id')
    .apply(lambda g: wmae(g['y'].values, g['y_pred'].values, g['is_holiday'].values))
    .reset_index()
    .rename(columns={0: 'wmae'})
    .sort_values('wmae', ascending=False)
)
wandb.log({'per_series_wmae': wandb.Table(dataframe=per_series)})

wandb.finish()

## 3. Save Best Model to MLflow Registry

In [ ]:
import dagshub
dagshub.init(repo_owner='ikvas22', repo_name='Walmart-Recruiting---Store-Sales-Forecasting', mlflow=True)

In [ ]:
mlflow.set_experiment(MLFLOW_EXPERIMENT)

In [ ]:
class TFTWrapper(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        with open(context.artifacts['nf_model'], 'rb') as f:
            self.nf = pickle.load(f)

    def predict(self, context, model_input: pd.DataFrame) -> pd.DataFrame:
        """
        Accepts raw merged DataFrame with:
        [Store, Dept, Date, Weekly_Sales, MarkDown1-5,
         Temperature, Fuel_Price, CPI, Unemployment, IsHoliday]
        Returns [Store, Dept, Date, Weekly_Sales_pred].
        """
        df_in = model_input.copy()
        df_in['Date']         = pd.to_datetime(df_in['Date'])
        df_in['unique_id']    = df_in['Store'].astype(str) + '_' + df_in['Dept'].astype(str)
        df_in['week']         = df_in['Date'].dt.isocalendar().week.astype(int)
        df_in['month']        = df_in['Date'].dt.month
        df_in['is_holiday']   = df_in['IsHoliday'].astype(int)
        markdown_cols = ['MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5']
        df_in[markdown_cols]  = df_in[markdown_cols].fillna(0)
        # CPI / Unemployment nulls — same fill logic as preprocessing
        df_in[['CPI', 'Unemployment']] = (
            df_in.groupby('Store')[['CPI', 'Unemployment']]
            .transform(lambda x: x.ffill().bfill())
        )
        df_in[['CPI', 'Unemployment']] = df_in[['CPI', 'Unemployment']].fillna(
            df_in[['CPI', 'Unemployment']].median()
        )

        keep = ['unique_id', 'Date', 'Weekly_Sales',
        'MarkDown1','MarkDown2','MarkDown3','MarkDown4','MarkDown5',
        'Temperature','Fuel_Price','CPI','Unemployment',
        'week','month','is_holiday']
        df_in = df_in[[c for c in keep if c in df_in.columns]]


        df_nf_in = (
            df_in
            .rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
            .sort_values(['unique_id', 'ds'])
        )

        # ── Build futr_df: future exog features for the H forecast weeks ──
        # For each unique_id, generate the next H weekly dates after its
        # last observed date and compute calendar features for them.
        # Calendar features are deterministic — derivable from dates alone —
        # which is exactly why they are valid future exogenous inputs.
        last_dates = df_nf_in.groupby('unique_id')['ds'].max().reset_index()
        futr_rows = []
        for _, row in last_dates.iterrows():
            future_dates = pd.date_range(
                start   = row['ds'] + pd.Timedelta(weeks=1),
                periods = self.nf.h,
                freq    = 'W-FRI',
            )
            for d in future_dates:
                futr_rows.append({'unique_id': row['unique_id'], 'ds': d})
        futr_df = pd.DataFrame(futr_rows)
        futr_df['week']       = futr_df['ds'].dt.isocalendar().week.astype(int)
        futr_df['month']      = futr_df['ds'].dt.month
        # Walmart holiday weeks: Super Bowl (6), Labor Day (36),
        # Thanksgiving (47), Christmas (52)
        holiday_weeks = {6, 36, 47, 52}
        futr_df['is_holiday'] = futr_df['week'].isin(holiday_weeks).astype(int)

        preds    = self.nf.predict(df=df_nf_in, futr_df=futr_df)
        preds    = preds.reset_index()   # unique_id comes back as index
        pred_col = [c for c in preds.columns if c not in ['unique_id', 'ds']][0]
        preds    = preds.rename(columns={pred_col: 'Weekly_Sales_pred'})
        preds[['Store', 'Dept']] = preds['unique_id'].str.split('_', expand=True).astype(int)
        return preds[['Store', 'Dept', 'ds', 'Weekly_Sales_pred']].rename(columns={'ds': 'Date'})


os.makedirs('mlflow_artifacts', exist_ok=True)
model_path = 'mlflow_artifacts/nf_auto_tft.pkl'
with open(model_path, 'wb') as f:
    pickle.dump(nf_auto, f)

with mlflow.start_run(run_name='TFT_Best_Model') as run:
    mlflow.log_params(best_params)
    mlflow.log_metrics({
        'val_wmae'         : best_wmae,
        'val_mae'          : best_mae,
        'baseline_wmae'    : baseline_wmae,
        'wmae_improvement' : baseline_wmae - best_wmae,
        'n_trials'         : N_TRIALS,
    })
    mlflow.pyfunc.log_model(
        artifact_path         = 'tft_model',
        python_model          = TFTWrapper(),
        artifacts             = {'nf_model': model_path},
        registered_model_name = MLFLOW_MODEL_NAME,
    )
    print(f'Registered: {MLFLOW_MODEL_NAME}')
    print(f'Run ID    : {run.info.run_id}')
    print(f'Best WMAE : {best_wmae:,.2f}')

In [ ]:
loaded = mlflow.pyfunc.load_model(f'models:/{MLFLOW_MODEL_NAME}/latest')
# TFT needs the full merged dataframe including covariates
sample = df[df['Store'] == 1].head(100).copy()
result = loaded.predict(sample)
print('Registry load OK')
print(result.head())